# W06A · B · U-Net을 학습하고 생성·비교 웹앱 만들기
2026-10-05 · 주교재4.1–4.7 · CPU · Python3.12/3.13

실제 구조·가중치 갱신을 확인하고 숫자5 프로젝트를 실행합니다. 기존 4000회 학습 모델의 전체 숫자 생성과 구별하며, 마지막에 세 가지 확산 실험을 웹앱에 연결합니다.

**패키지0.1.9 / 2026-fall-w06a**. 새 Colab 사본을 Drive에 저장하고 첫 셀부터 실행하세요. MNIST 약11.6 MB와 패키지를 자동 준비합니다. API 키·개인 파일 업로드·CLIP 가중치 다운로드는 필요 없습니다.

[기초](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/course/notion/w06a/w06a-foundations.md) · [설계 선택](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/course/notion/w06a/w06a-design.md) · [프로젝트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/course/notion/w06a/w06a-project.md) · [기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/course/notion/w06a/w06a-workbook.md) · [전체 API 정의](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/W06A-API.md)


## 0. 환경 준비
설치 전에 이미 모듈을 import했다면 런타임을 재시작하세요. 설치·다운로드·계산의 시간을 구별합니다. 실패를 가짜 출력으로 대체하지 않습니다.


In [ ]:
import sys,subprocess,os,time
from pathlib import Path
from importlib.metadata import version,PackageNotFoundError
PUBLIC_REF='2026-fall-w06a'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.9'),('transformers','5.15.1'),('diffusers','0.40.0'),('gradio','6.26.0')])
except PackageNotFoundError:
    ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','torch','diffusers','gradio','transformers')):
        raise RuntimeError('이미 import한 패키지의 버전을 바꾸려면 런타임을 재시작한 뒤 첫 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
print('Python',sys.version.split()[0],'package',version('luna-genai'),'diffusers',version('diffusers'))
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(2)
started=time.perf_counter()


## 1. 자동 데이터 준비
[`autoencoder.load_mnist`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/autoencoder.py#L92)는 공식 MNIST를 다운로드하고 체크섬을 검사합니다. 반환 이미지는 CPU float32 `(N,1,28,28)`의 [0,1] 값입니다. `2*x-1`로 [-1,1]을 만듭니다. train/validation은 같은 공식 학습 집합에서 겹치지 않게 선택하고 test는 별도 공식 테스트 집합입니다. 반환 객체에 이미지·테스트 라벨·분할 인덱스·이번 다운로드 목록이 들어 있습니다. 호출은 파일 캐시를 만들 수 있지만 모델을 학습하지 않습니다.


In [ ]:
from luna_genai.autoencoder import load_mnist
data=load_mnist(train_size=12000,validation_size=1000,test_size=1000,seed=1337)
clean=data.test[:1]*2-1
print('다운로드:',data.downloads or '없음: 검증된 캐시 재사용')
print('train/validation/test:',len(data.train),len(data.validation),len(data.test))
print('train/validation 중복:',len(np.intersect1d(data.train_ids,data.validation_ids)))
print('원본 텐서:',clean.shape,clean.dtype,'범위',float(clean.min()),float(clean.max()))


## 2. 새 모델과 학습된 모델
[`diffusion.TinyTimeUNet`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L85)은 seed에 따른 새 가중치를 만듭니다. [`diffusion.TinyTimeUNet.forward`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L104)는 CPU float32 `(B,1,28,28)` noisy 입력과 int64 `(B,)` 시점에서 같은 shape의 잡음을 예측합니다. 학습하거나 파일을 읽는 메소드가 아닙니다.
[`diffusion.load_diffusion_checkpoints`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L204)는 패키지 NPZ를 SHA-256 검사 후 읽어 모델 사전과 학습 기록을 반환합니다. 로드는 학습이 아닙니다. [`diffusion_lab.trace_unet`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab.py#L113)은 임시 hook으로 각 block의 실제 shape를 관찰하며 모수와 원래 train/eval 상태를 보존합니다.


In [ ]:
from luna_genai.diffusion import (TinyTimeUNet,make_scheduler,forward_noise,training_step,evaluate_noise,load_diffusion_checkpoints,image_grid)
from luna_genai.diffusion_lab import trace_unet,select_digit,fit_diffusion_project,sample_with_sampler
from luna_genai.diffusion_lab_app import build_diffusion_lab,noise_view,target_view,generation_view,APP_CSS
models,record=load_diffusion_checkpoints()
fresh=TinyTimeUNet(seed=20261005)
batch=data.train[:8]*2-1;g=torch.Generator().manual_seed(23)
t=torch.randint(100,(len(batch),),generator=g);epsilon=torch.randn(batch.shape,generator=g)
noisy=forward_noise(batch,t,epsilon)
display(trace_unet(fresh,noisy,t))
print('모수 수:',sum(p.numel() for p in fresh.parameters()))
display(pd.DataFrame([dict(name=k,step=v['step'],ema=v['ema']) for k,v in record['checkpoints'].items()]))


### B1 · dec2의 입력이96채널인 이유를 설명하세요. 정답 epsilon을 model.forward에 함께 넣어도 되나요?

목표: 결과를 먼저 예상하고 근거를 설명합니다. 힌트는 앞의 식과 실제 출력입니다. 연습용 변수의 빈칸은 기본 실행을 막지 않습니다.


## 3. 한 번의 실제 갱신
[`diffusion.training_step`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L122)은 모델·optimizer·원본·시점·정답 잡음을 받아 MSE, backward, Adam 갱신을 수행합니다. 모델과 optimizer 상태를 바꾸고, loss·gradient·갱신 전후 변화량을 반환합니다. [`diffusion.forward_noise`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L49)는 고정100단계 cosine 전방 계산이며 학습하지 않습니다.


In [ ]:
optimizer=torch.optim.Adam(fresh.parameters(),lr=.001)
step_result=training_step(fresh,optimizer,batch,t,epsilon)
display(pd.DataFrame([step_result]))
my_update_explanation=''


### B2 · backward_parameter_change가0인데 parameter_change가 양수인 것은 오류인가요?

목표: 결과를 먼저 예상하고 근거를 설명합니다. 힌트는 앞의 식과 실제 출력입니다. 연습용 변수의 빈칸은 기본 실행을 막지 않습니다.


## 4. 숫자5 비조건부 프로젝트
[`diffusion_lab.select_digit`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab.py#L140)은 이미 준비한 분할을 각각 필터합니다. `digit=None`은 전체, 0..9는 한 숫자이며, [-1,1] 이미지·원래 인덱스·개수를 반환합니다. 라벨은 모집단 선택에만 사용합니다.
[`diffusion_lab.fit_diffusion_project`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab.py#L167)은 **새 모델**과 Adam을 만들고 지정한 횟수만큼 실제 갱신합니다. 기본32회, batch16, lr0.001, seed20261005입니다. 이미지 선택은 복원 추출입니다. 반환값은 학습 모델과 step별 DataFrame입니다. 기존 체크포인트는 바뀌지 않습니다.
[`diffusion.evaluate_noise`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L148)는 고정 이미지·잡음·시점별 MSE와0예측 기준선을 반환합니다. 모수를 바꾸지 않습니다. 프로젝트 비교에는 validation을 사용합니다.


In [ ]:
project=select_digit(data,5)
print('프로젝트 숫자:',project['digit'],'분할 개수:',project['counts'])
project_initial=TinyTimeUNet(seed=20261005)
before=evaluate_noise(project_initial,project['validation'][:64])
project_model,history=fit_diffusion_project(project['train'],steps=32,batch_size=16,learning_rate=.001,seed=20261005)
after=evaluate_noise(project_model,project['validation'][:64])
display(history.iloc[[0,-1]])
display(before[['t','mse']].rename(columns={'mse':'before'}).merge(after[['t','mse']].rename(columns={'mse':'after'}),on='t'))
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
axes[0].plot(history.step,history.loss);axes[0].set(xlabel='Step',ylabel='Training epsilon MSE')
axes[1].plot(before.t,before.mse,'o-',label='before');axes[1].plot(after.t,after.mse,'o-',label='after32');axes[1].legend();axes[1].set(xlabel='Code t',ylabel='Fixed validation MSE');plt.show()


### B3 · steps=32를32epoch라고 적어도 되나요? 이 모델과 기존 trained 체크포인트의 모집단은 같은가요?

목표: 결과를 먼저 예상하고 근거를 설명합니다. 힌트는 앞의 식과 실제 출력입니다. 연습용 변수의 빈칸은 기본 실행을 막지 않습니다.


### 선택 확장: 수업 후 학습량 늘리기
아래 기본값0은 긴 추가 학습을 요청하지 않은 상태입니다. 1000 등으로 직접 바꾸면 새 프로젝트 모델을 처음부터 학습합니다. CPU 소요 시간과 validation을 기록하고 기존 모델과 구별하세요. 기본 검증은 위32회 실제 학습을 대상으로 합니다.


In [ ]:
LONG_STEPS=0
if LONG_STEPS>0:
    long_model,long_history=fit_diffusion_project(project['train'],steps=LONG_STEPS,batch_size=32)
    display(long_history.tail())
else:
    print('선택 장시간 훈련은 실행하지 않았습니다. 위32회 실제 훈련과는 별개입니다.')


## 5. 같은 모델의 생성 방법 비교
[`diffusion_lab.sample_with_sampler`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab.py#L200)은 원본 없이 새 Gaussian에서 시작합니다. DDPM은100회, DDIM은2..100회이며 기본20회입니다. DDIM eta0·trailing 간격으로 원래 학습된 beta와 epsilon 목표를 유지합니다. 반환값에는 이미지·초기 잡음·중간 상태·원본 추정·호출 수·소요 시간이 있습니다. 모수와 원래 모드는 보존됩니다.
[`diffusion.image_grid`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion.py#L225)는 [-1,1] 이미지 배치에서 matplotlib Figure를 반환합니다. 표시용 자르기는 텐서를 바꾸지 않습니다.


In [ ]:
results={};sampling_rows=[]
for name,method,steps in [('initial','DDPM',100),('trained','DDPM',100),('trained','DDIM',20)]:
    result=sample_with_sampler(models[name],method=method,steps=steps,count=8,seed=42)
    results[(name,method)]=result
    fig=image_grid(result['images'],title=f'{name} / {method}{steps} / seed42');display(fig);plt.close(fig)
    sampling_rows.append(dict(model=name,method=method,steps=steps,seconds=result['seconds']))
display(pd.DataFrame(sampling_rows))
fixed_test=evaluate_noise(models['trained'],data.test[:64]*2-1)
display(fixed_test)


### B4 · 추론 스케줄을 linear로 교체하면 공정한 스케줄 품질 비교인가요? DDIM20과100의 속도·품질은 어떻게 보고해야 하나요?

목표: 결과를 먼저 예상하고 근거를 설명합니다. 힌트는 앞의 식과 실제 출력입니다. 연습용 변수의 빈칸은 기본 실행을 막지 않습니다.


## 6. 마지막 웹앱을 직접 확장하기
[`diffusion_lab_app.build_diffusion_lab`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab_app.py#L95)에 모델 사전을 주면 노이즈·목표·생성 세 탭을 구성합니다. [`diffusion_lab_app.generation_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06a/src/luna_genai/diffusion_lab_app.py#L71)는 학습 상태·샘플러·seed를 받아 실제 생성 격자와 중간 경로·측정값을 반환합니다.
아래에서는 자신의 함수와 입력·출력을 연결한 실험 기록 패널을 덧붙입니다. 기본 함수는 작성 안내만 반환합니다. 함수 본문을 수정한 뒤 실제 버튼으로 결과를 확인하세요. 핵심 생성 탭은 이 빈칸과 독립적입니다.


In [ ]:
import gradio as gr
def my_experiment_summary(setting,observation):
    # TODO: 설정과 관찰을 구분하고, 해석의 한계를 한 문장 덧붙이세요.
    return '설정과 관찰을 읽어 자신의 실험 요약을 반환하도록 이 함수를 수정하세요.'


### B5 · my_experiment_summary를 완성하고 입력 위젯 두 개를 올바른 순서로 연결하세요. 출력에 실제 관찰과 해석의 한계를 구별하세요.

목표: 결과를 먼저 예상하고 근거를 설명합니다. 힌트는 앞의 식과 실제 출력입니다. 연습용 변수의 빈칸은 기본 실행을 막지 않습니다.


In [ ]:
app=build_diffusion_lab(data.test[:32]*2-1,models)
with app:
    gr.Markdown('## 내 실험 기록 · 자신의 함수로 입력과 출력을 연결하기')
    setting_text=gr.Textbox(label='내가 바꾼 설정',value='trained / DDIM20 / seed42')
    observation_text=gr.Textbox(label='실제로 본 결과',placeholder='직접 관찰한 뒤 작성')
    summarize=gr.Button('내 함수로 기록 정리')
    summary=gr.Textbox(label='설정·관찰·해석의 한계',lines=4)
    summarize.click(my_experiment_summary,[setting_text,observation_text],summary,api_name='my_summary')
_,_,callback_noise=noise_view(0,49,42,'cosine',1.,images=data.test[:32]*2-1)
fig,callback_target=target_view(0,49,42,'v_prediction',images=data.test[:32]*2-1);plt.close(fig)
fig,trajectory,callback_generation=generation_view('trained','DDIM20',42,models=models);plt.close(fig);plt.close(trajectory)


In [ ]:
assert not np.intersect1d(data.train_ids,data.validation_ids).size
assert len(history)==32 and np.isfinite(history.loss).all()
assert (history.gradient_norm>0).all() and (history.parameter_change>0).all()
assert (history.backward_parameter_change==0).all()
assert (fixed_test.mse<fixed_test.zero_mse).all()
assert results[('trained','DDPM')]['images'].shape==(8,1,28,28)
assert torch.isfinite(results[('trained','DDIM')]['images']).all()
assert callback_noise['t']==49 and callback_target['max_abs_error']<1e-5
assert callback_generation['steps']==20 and len(app.config['dependencies'])==4
print('W06A B 검증 통과 · 계산 구간',round(time.perf_counter()-started,2),'초')


In [ ]:
if IN_COLAB:
    app.launch(share=True,inline=False,prevent_thread_lock=True,css=APP_CSS)
else:
    print('로컬에서는 app.launch(css=APP_CSS)로 실행합니다. Colab 공유 URL은 런타임 동안만 유지됩니다.')


공유 링크를 새 탭에서 열고 세 탭의 버튼과 자신의 기록 버튼을 각각 실행하세요. 학습 상태·샘플러·seed와 실제 관찰을 기록합니다. 모델 호출 횟수, 예측 MSE, 생성 품질은 서로 다른 값입니다. 마지막으로 기록지의12개 질문에 답한 뒤 해설을 확인하세요.
